# 04 — YOLOv11 Training

Trains a YOLOv11 variant on the unified dataset (notebook 02, health-checked in 03). Default is `yolo11s` — a good balance for an 800-image dataset.

| Variant   | Params | Speed   | When to pick |
|-----------|-------:|---------|--------------|
| `yolo11n` | ~2.6 M | fastest | edge / demo |
| `yolo11s` | ~9.4 M | fast    | **default** |
| `yolo11m` | ~20 M  | moderate| if accuracy plateaus |

In [5]:
# Install dependencies
# Ultralytics pulls in torch, torchvision, opencv-python, numpy, pandas, matplotlib.
%pip install -q "ultralytics==8.3.*" pandas matplotlib
# On a bare Linux box (no system libGL) OpenCV import can fail — if so, uncomment:
# !apt-get update -qq && apt-get install -y -qq libgl1

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Reinstall PyTorch with CUDA support — force-reinstall ensures the CUDA build
# wins even if ultralytics already pulled in a +cpu wheel above.
# cu126 wheels are compatible with CUDA driver 12.x and 13.x (RTX 4060, etc.)
%pip install -q --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

In [ ]:
# Config — paths, training hyperparameters
DATA_YAML   = Path('../data/dataset/data.yaml').resolve()  # built by notebook 02
RUNS_DIR    = Path('../runs/detect')                       # ultralytics writes here
WEIGHTS_DIR = Path('../weights')                           # promoted best.pt lands here

# Training config — change `model` to compare yolo11n / yolo11s / yolo11m
CFG = dict(
    model='yolo11n.pt',
    data=str(DATA_YAML),
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer='SGD',
    lr0=0.01,
    patience=20,            # early stopping
    device=0,               # explicitly use GPU 0; ultralytics falls back to CPU silently without this
    project=str(RUNS_DIR),
    name='campus_yolo11s',
    seed=42,
)

assert DATA_YAML.exists(), f'missing {DATA_YAML} — run notebook 02 first'
assert torch.cuda.is_available(), 'CUDA not available — restart kernel after the torch CUDA install cell'
WEIGHTS_DIR.mkdir(exist_ok=True)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))
CFG

In [ ]:
# Config — paths, training hyperparameters
DATA_YAML   = Path('../data/dataset/data.yaml').resolve()  # built by notebook 02
RUNS_DIR    = Path('../runs/detect')                       # ultralytics writes here
WEIGHTS_DIR = Path('../weights')                           # promoted best.pt lands here

# Training config — change `model` to compare yolo11n / yolo11s / yolo11m
CFG = dict(
    model='yolo11s.pt',
    data=str(DATA_YAML),
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer='SGD',
    lr0=0.01,
    patience=15,            # early stopping
    project=str(RUNS_DIR),
    name='campus_yolo11s',
    seed=42,
)

assert DATA_YAML.exists(), f'missing {DATA_YAML} — run notebook 02 first'
WEIGHTS_DIR.mkdir(exist_ok=True)
print('CUDA available:', torch.cuda.is_available())
CFG

CUDA available: False


{'model': 'yolo11s.pt',
 'data': 'C:\\Users\\mitah\\github_projects\\ai_cv_project\\data\\dataset\\data.yaml',
 'epochs': 100,
 'imgsz': 640,
 'batch': 16,
 'optimizer': 'SGD',
 'lr0': 0.01,
 'patience': 20,
 'project': '..\\runs\\detect',
 'name': 'campus_yolo11s',
 'seed': 42}

# Train YOLOv11 with the configured hyperparameters
import time

model = YOLO(CFG['model'])

_t0 = time.time()
results = model.train(**{k: v for k, v in CFG.items() if k != 'model'})
_elapsed = time.time() - _t0

run_dir = Path(results.save_dir)
print(f'run dir: {run_dir}')
print(f'Total training time: {int(_elapsed // 3600)}h {int(_elapsed % 3600 // 60)}m {int(_elapsed % 60)}s')

In [4]:
# Train YOLOv11 with the configured hyperparameters
model = YOLO(CFG['model'])
results = model.train(**{k: v for k, v in CFG.items() if k != 'model'})
run_dir = Path(results.save_dir)
print('run dir:', run_dir)

New https://pypi.org/project/ultralytics/8.4.41 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.253  Python-3.13.12 torch-2.11.0+cpu CPU (12th Gen Intel Core i5-12400F)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\mitah\github_projects\ai_cv_project\data\dataset\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=False

KeyboardInterrupt: 

## 2. Training curves
Loss + validation mAP per epoch from `results.csv`.

In [ ]:
# Plot loss + mAP curves from results.csv
df = pd.read_csv(run_dir / 'results.csv')
df.columns = [c.strip() for c in df.columns]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df.plot(x='epoch', y=['train/box_loss', 'train/cls_loss', 'val/box_loss', 'val/cls_loss'], ax=ax[0])
ax[0].set_title('Loss curves')
df.plot(x='epoch', y=['metrics/mAP50(B)', 'metrics/mAP50-95(B)'], ax=ax[1])
ax[1].set_title('Validation mAP')
plt.tight_layout(); plt.show()

## 3. Promote best checkpoint
Copy `best.pt` from the run directory to `weights/best.pt` for downstream notebooks.

In [ ]:
# Copy best.pt → weights/best.pt
best = run_dir / 'weights' / 'best.pt'
dst = WEIGHTS_DIR / 'best.pt'
shutil.copy2(best, dst)
print('saved', dst)

## 4. Quick sanity inference

In [ ]:
# Predict on a few test images, save annotated outputs under <run_dir>/sanity/
test_imgs = list((Path('../data/dataset/images/test')).glob('*.jpg'))[:4]
m = YOLO(dst)
res = m.predict(source=[str(p) for p in test_imgs], imgsz=CFG['imgsz'], conf=0.25,
                save=True, project=str(run_dir), name='sanity')
print('predictions saved under', run_dir / 'sanity')

### Expected training indicators
- `val/box_loss` decreases smoothly and plateaus
- `mAP@0.5` > 0.80 on val for all 4 classes within ~60–80 epochs
- No runaway gap between train and val loss

Proceed to **notebook 05** for full evaluation.